# QFF 2026 Phase 1 — PS-O1: Max-Cut on Weighted Graphs

This notebook runs the reproducible Max-Cut experiment for Processor A (5 qubits, linear chain) and Processor B (9 qubits, participant-designed 12-edge graph).

**Recommended environment:** local Python 3.12 with dependencies installed from `requirements.txt`. Browser-only notebook runtimes may not support Qiskit Aer or compiled dependencies.

**Main research question:** how do coupling-map choices change routing, circuit depth/CX count, and the quality of noisy QAOA samples?

The official guide specifies the six-node graph generator, integer weights from 1 to 5, QAOA p=1 or p=2, the processor budgets, transpilation settings, and synthetic noise rates. Since it doesn't provide a random seed for edge weights, this project uses NumPy seed 42 and records that choice.

## 1. Imports and project folders

Run the notebook with the project root as the working directory. The modular implementation is under `src/`; the cells below call those functions so the experiment remains reproducible.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / "processors").exists() or not (ROOT / "src").exists():
    raise FileNotFoundError("Open/run this notebook from the PS_O1_MaxCut_QAOA_Starter project folder.")

for package in ["numpy", "pandas", "networkx", "matplotlib", "scipy", "qiskit", "qiskit-aer"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

from src.problem import build_weighted_graph, graph_edge_rows, enumerate_classical, save_graph_data
from run_experiment import ensure_dirs, initial_classical_stage, run_qaoa_stage
ensure_dirs()


## 2. Generate the weighted graph

The graph structure follows `networkx.erdos_renyi_graph(n=6, p=0.5, seed=42)`. Edge weights are integers from 1 through 5 generated using a documented NumPy seed of 42. This makes our instance reproducible while being transparent that the guide doesn't prescribe the weight seed.

In [ ]:
graph = build_weighted_graph()
save_graph_data(graph, ROOT)
edge_table = pd.DataFrame(graph_edge_rows(graph))
display(edge_table)
print(f"Nodes: {graph.number_of_nodes()} | Edges: {graph.number_of_edges()}")


## 3. Establish the exact classical baseline

Max-Cut has a group-label symmetry: swapping 0 and 1 on every node doesn't change which edges cross. We can fix node 0 to 0 and search the remaining five bits. That leaves 2⁵ = 32 representatives, so exact brute force is simple and verifies that the problem fits Processor A's five-qubit budget.

In [ ]:
optimum_table, classical_optimum, optimal_assignments = enumerate_classical(graph)
classical_assignment = optimal_assignments[0]
print(f"Exact maximum cut value: {classical_optimum}")
print(f"One optimal node assignment (nodes 0..5): {''.join(map(str, classical_assignment))}")
display(optimum_table.head(10))
optimum_table.to_csv(ROOT / "results" / "tables" / "classical_all_cuts.csv", index=False)


## 4. View the problem and exact cut

In these 2D diagrams, each node is a graph vertex, each edge label is its weight, and gold edges cross the partition. The classical optimum gives an exact reference for judging the QAOA output.

In [ ]:
# This call writes the graph and architecture figures as well as the classical tables.
classical_optimum, classical_assignment = initial_classical_stage(graph)
display(Image(filename=str(ROOT / "results" / "figures" / "01_original_weighted_graph.png")))
display(Image(filename=str(ROOT / "results" / "figures" / "02_classical_optimum_cut.png")))


## 5. Processor A and Processor B connectivity

- **Processor A:** the five-qubit linear-chain baseline.
- **Processor B:** a participant-designed nine-qubit graph with twelve undirected edges. Its first five physical qubits include the six pairwise couplings required by this specific symmetry-reduced graph. Four additional qubits form a sparse extension.

Processor B is therefore deliberately tailored to this single instance; that design choice should be discussed honestly as a limitation in the report.

In [ ]:
display(Image(filename=str(ROOT / "results" / "figures" / "architecture_processor_A.png")))
display(Image(filename=str(ROOT / "results" / "figures" / "architecture_processor_B.png")))


## 6. Run QAOA and benchmark both processors

The first run uses depth `p=1` and 4096 shots. QAOA parameters are optimized once using exact statevector expectation values, then the same logical parameters are compiled and sampled on both processor models. Both receive the same synthetic noise rates from the guide.

This cell requires Qiskit and Qiskit Aer. If you have not installed the dependencies locally, run `python -m pip install -r requirements.txt` in the project's virtual environment first.

In [ ]:
DEPTH_P = 1  # The guide allows p=1 or p=2.
SHOTS = 4096
SEED = 42
run_qaoa_stage(graph, classical_optimum, classical_assignment, depth=DEPTH_P, shots=SHOTS, seed=SEED)


## 7. Read the benchmark table

Use the actual result CSV created by the previous cell. Do not insert fabricated values into the report. The SWAP count is measured in a routing-stage diagnostic before the final native-gate translation; the additional CX count versus an unconstrained baseline is only a proxy for routing cost.

In [ ]:
comparison_path = ROOT / "results" / "tables" / f"processor_comparison_p{DEPTH_P}.csv"
comparison = pd.read_csv(comparison_path)
display(comparison)


## 8. Inspect the generated figures

The script creates the optimizer trace, a side-by-side processor metric chart, a noisy output cut distribution, and a classical-versus-QAOA cut comparison. These are 2D scientific visualizations; the project does not require 3D plots.

In [ ]:
figure_names = [
    f"qaoa_convergence_p{DEPTH_P}.png",
    f"processor_comparison_p{DEPTH_P}.png",
    f"noisy_cut_distribution_p{DEPTH_P}.png",
    f"classical_vs_qaoa_cut_p{DEPTH_P}.png",
]
for name in figure_names:
    path = ROOT / "results" / "figures" / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))


## 9. Discussion prompts

1. Did QAOA sample the exact optimal cut? What was the optimal-cut shot rate?
2. Which architecture produced the lower circuit depth and CX count?
3. Did routing-stage SWAP counts align with the CX-count difference?
4. How did noise change the sample mean and optimal-cut shot rate?
5. Since Processor B is tailored to this graph, what can—and cannot—be concluded from the comparison?

Add the actual numbers and explanations to `report.md`. Save package versions and any changed seeds/settings so someone else can reproduce the work.